[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/simon-clematide/gemma-qlora-finetuning/blob/main/notebooks/11_colab_l4_live_demo.ipynb)

# `11_colab_l4_live_demo.ipynb` — End-to-End Live QLoRA / LoRA Fine-Tuning & Test-Set Evaluation on Google Colab (L4 / T4 GPU)

This self-contained notebook is designed to run **live on a Google Colab L4 GPU (`24 GB VRAM`) or free T4 GPU (`15 GB VRAM`)** (or locally on CUDA / Apple Silicon `mps`).
- **Zero live XML preprocessing needed:** The repository includes all 26,209 canonical GermEval 2017 documents in compact `data/canonical/*.jsonl.gz` archives (`5 MB` total) and automatically materializes the Task C and Task D JSONL files in ~1.5 seconds.
- **Configurable 4-bit NF4 QLoRA or 16-bit `bfloat16` LoRA:** Defaults to **4-bit NF4 QLoRA** (`USE_4BIT_QUANTIZATION = True` in Step 3), and can be switched to **16-bit `bfloat16` LoRA** (`USE_4BIT_QUANTIZATION = False`) with a single boolean toggle.
- **100% Pure-Python Evaluation:** Computes Strict Task C1/C2 Micro-F1, Macro-F1, Exact-Set Accuracy, Bootstrap 95% CIs, and Exact Official Java-Compatible Scores in pure Python (`src/germeval_llm/evaluation.py`).

---
## Step 1: Google Colab Bootstrap (Clone Repo, Install Package, Authenticate HF)

In [ ]:
import importlib, os, subprocess, sys
from pathlib import Path

# If running on Google Colab, clone or sync the latest repo and install in editable mode (-e):
if 'google.colab' in sys.modules:
    REPO_URL = os.environ.get('GERMEVAL_REPO_URL', 'https://github.com/simon-clematide/gemma-qlora-finetuning.git')
    if not Path('pyproject.toml').exists() and not Path('gemma-qlora-finetuning/pyproject.toml').exists():
        print(f'Cloning {REPO_URL}...')
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    if Path('gemma-qlora-finetuning/pyproject.toml').exists():
        os.chdir('gemma-qlora-finetuning')
    subprocess.run(['git', 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'bitsandbytes>=0.46.1', '-e', '.[ml,cuda]'], check=True)

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import germeval_llm.convert
import germeval_llm.inference
import germeval_llm.training
importlib.reload(germeval_llm.convert)
importlib.reload(germeval_llm.inference)
importlib.reload(germeval_llm.training)
from germeval_llm.convert import ensure_derived_datasets_ready
from germeval_llm.inference import detect_compute_device

ensure_derived_datasets_ready()
hw = detect_compute_device()
git_rev = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
print(f'Repository initialized: {ROOT.name} (commit {git_rev})')
print('Detected compute hardware:', hw)

### Authenticate with Hugging Face Hub (for gated `google/gemma-2-2b-it`)
Run `login(skip_if_logged_in=False)` below and paste your Hugging Face access token (`skip_if_logged_in=False` forces a fresh prompt even if an older token is cached in the runtime):

In [ ]:
import os
from huggingface_hub import login

os.environ.pop('HF_TOKEN', None)
os.environ.pop('HUGGING_FACE_HUB_TOKEN', None)
try:
    login(skip_if_logged_in=False)
except TypeError:
    login(new_session=True)

---
## Step 2: Inspect the Dataset & Completion-Only Loss Masking
Before loading the model, let's inspect a real training example and verify how our prompt and completion are formatted for `google/gemma-2-2b-it`. Remember:
- **Task C Output Contract:** `{"labels": ["Category:polarity", ...]}` (or `{"labels": []}` when `relevance=false`).
- **Completion-Only Loss (`labels = -100`):** During LoRA / QLoRA training, all prompt tokens up to `<start_of_turn>model\n` are masked with `-100` so gradients only update on the JSON completion.

In [ ]:
import json
import pandas as pd
from IPython.display import display
from germeval_llm.xml_parser import load_canonical_jsonl
from germeval_llm.schemas import TaskCExample
from germeval_llm.prompts import format_gemma_chat_string, compute_completion_only_labels

train_docs = load_canonical_jsonl('data/canonical/train.jsonl')
dev_docs = load_canonical_jsonl('data/canonical/dev.jsonl')
test_syn_docs = load_canonical_jsonl('data/canonical/test_syn.jsonl')
test_dia_docs = load_canonical_jsonl('data/canonical/test_dia.jsonl')
print(f'Loaded splits -> train: {len(train_docs)}, dev: {len(dev_docs)}, test_syn: {len(test_syn_docs)}, test_dia: {len(test_dia_docs)}')

with Path('data/derived/task_c/train.jsonl').open() as f:
    sample_ex = TaskCExample.model_validate_json(f.readline())

print('\nSample Document Text:', sample_ex.text)
print('Target JSON Completion:', sample_ex.completion[0].content)

---
## Step 3: Configure Experiment Scope & Precision (`4-bit NF4 QLoRA` vs. `16-bit bfloat16 LoRA`)
- **`USE_4BIT_QUANTIZATION = True` (Default — 4-bit `nf4` QLoRA via `configs/qlora.yaml`):**
  Compresses frozen base weights by ~72% (`~1.4 GB` VRAM, `~2.9 GB` peak training VRAM) via `bitsandbytes` 4-bit NormalFloat quantization with double quantization (`bnb_4bit_use_double_quant=True`). Works on both Colab **L4 (`24 GB`)** and **T4 (`15 GB`)** GPUs.
- **`USE_4BIT_QUANTIZATION = False` (16-bit `bfloat16` LoRA via `configs/lora.yaml`):**
  Uses native 16-bit `bfloat16` base weights (`~5.0 GB` VRAM, `~7.2 GB` peak training VRAM). Requires no `bitsandbytes` dependency and trains ~20% faster per step on an L4 GPU.
- **`RECOMPUTE_ZERO_SHOT_BASELINE = False` (Default — Instant Zero-Shot Baseline Reuse):**
  Loads the stored Zero-Shot `P1_guidelines` predictions from `results/predictions/P1_guidelines_0shot_test_syn.jsonl` and scores them on `eval_docs` in <0.2 seconds so you can jump straight into Step 5 (`SFTTrainer` fine-tuning). Set `RECOMPUTE_ZERO_SHOT_BASELINE = True` if you want to run live Zero-Shot inference on the GPU.

In [ ]:
import torch

# Set RUN_LIVE_GPU = True on Google Colab (automatically enabled when CUDA GPU is present!)
RUN_LIVE_GPU = torch.cuda.is_available()
USE_4BIT_QUANTIZATION = True           # True = 4-bit NF4 QLoRA (configs/qlora.yaml) | False = 16-bit bfloat16 LoRA (configs/lora.yaml)
RECOMPUTE_ZERO_SHOT_BASELINE = False   # False = reuse stored results/predictions/P1_guidelines_0shot_test_syn.jsonl (instant!)
DEMO_MODE = 'fast_l4'                  # Choose 'fast_l4' (~5 mins on L4) or 'full_l4' (~45 mins on L4)

METHOD_TAG = 'qlora_4bit' if USE_4BIT_QUANTIZATION else 'lora_16bit'
METHOD_LABEL = '4-bit NF4 QLoRA (r=16)' if USE_4BIT_QUANTIZATION else '16-bit bfloat16 LoRA (r=16)'
TRAIN_CONFIG = 'configs/qlora.yaml' if USE_4BIT_QUANTIZATION else 'configs/lora.yaml'
ADAPTER_DIR = f'models/adapters/colab_{METHOD_TAG}_task_c'
STORED_BASELINE_PRED_PATH = 'results/predictions/P1_guidelines_0shot_test_syn.jsonl'
LIVE_BASELINE_PRED_PATH = 'results/predictions/colab_baseline_p1_test_syn.jsonl'
FT_PRED_PATH = f'results/predictions/colab_{METHOD_TAG}_r16_test_syn.jsonl'

if DEMO_MODE == 'fast_l4':
    TRAIN_SUBSET_SIZE = 1000
    MAX_TRAIN_STEPS = 120
    EVAL_MAX_DOCS = 150
else:
    TRAIN_SUBSET_SIZE = None  # All 19,432 train docs
    MAX_TRAIN_STEPS = None    # Full 3 epochs (3,644 steps)
    EVAL_MAX_DOCS = None      # All 2,566 test_syn docs

print(f'RUN_LIVE_GPU={RUN_LIVE_GPU} | MODE={METHOD_LABEL} ({TRAIN_CONFIG}) | DEMO_MODE={DEMO_MODE}')
print(f'RECOMPUTE_ZERO_SHOT_BASELINE={RECOMPUTE_ZERO_SHOT_BASELINE} | ADAPTER_DIR={ADAPTER_DIR}')

---
## Step 4: Baseline Evaluation on `test_syn` (`test_TIMESTAMP1`) Before Fine-Tuning
By default (`RECOMPUTE_ZERO_SHOT_BASELINE = False`), we load the precomputed Zero-Shot (`P1_guidelines`) predictions on `test_syn` from `results/predictions/P1_guidelines_0shot_test_syn.jsonl` and score the exact `eval_docs` slice with our **pure-Python evaluator** (`evaluate_task_c_predictions`) in <0.2 seconds—saving several minutes of GPU inference before training. (Set `RECOMPUTE_ZERO_SHOT_BASELINE = True` in Step 3 if you want to recompute them live on the GPU.)

In [ ]:
from germeval_llm.inference import (
    load_gemma_model_and_tokenizer,
    build_hf_generator_fn,
    run_task_c_inference_with_generator,
    save_predictions_jsonl,
    load_predictions_jsonl,
)
from germeval_llm.evaluation import evaluate_task_c_predictions

eval_docs = test_syn_docs[:EVAL_MAX_DOCS] if EVAL_MAX_DOCS else test_syn_docs

if RUN_LIVE_GPU and RECOMPUTE_ZERO_SHOT_BASELINE:
    print('Running live Zero-Shot P1 inference on GPU...')
    base_model, tokenizer = load_gemma_model_and_tokenizer(
        model_id='google/gemma-2-2b-it',
        load_in_4bit=USE_4BIT_QUANTIZATION,
    )
    gen_fn = build_hf_generator_fn(base_model, tokenizer, max_new_tokens=256)
    baseline_preds = run_task_c_inference_with_generator(
        eval_docs,
        run_id='colab_baseline_p1',
        split='test_syn',
        generate_fn=gen_fn,
        prompt_key='p1_guidelines',
    )
    save_predictions_jsonl(baseline_preds, LIVE_BASELINE_PRED_PATH)
    baseline_metrics = evaluate_task_c_predictions(eval_docs, baseline_preds)
    del base_model
    torch.cuda.empty_cache()
else:
    print(f'Reusing stored Zero-Shot P1 predictions from {STORED_BASELINE_PRED_PATH} ({len(eval_docs)} docs)...')
    stored_preds_all = load_predictions_jsonl(STORED_BASELINE_PRED_PATH)
    baseline_preds = stored_preds_all[:len(eval_docs)]
    baseline_metrics = evaluate_task_c_predictions(eval_docs, baseline_preds)

print('Baseline (Zero-Shot P1) Summary:')
print(json.dumps({
    'num_documents': baseline_metrics['num_documents'],
    'schema_validity_rate': baseline_metrics['schema_validity_rate'],
    'valid_exact_rate': baseline_metrics['valid_exact_rate'],
    'parse_status_counts': baseline_metrics['parse_status_counts'],
    'c1_strict_micro_f1': round(baseline_metrics['task_c1_category_strict']['micro_f1'], 4),
    'c2_strict_micro_f1': round(baseline_metrics['task_c2_category_sentiment_strict']['micro_f1'], 4),
    'c2_java_compat_micro_f1': round(baseline_metrics['task_c2_category_sentiment_java_compat']['micro_f1'], 4),
}, indent=2))

---
## Step 5: Live QLoRA / LoRA Fine-Tuning (`trl.SFTTrainer`)
Now we train our PEFT adapter ($r=16, \alpha=32$, targeting all 7 linear projections `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`) using Hugging Face TRL's **`SFTTrainer`** (`Supervised Fine-Tuning Trainer`) with `completion_only_loss=True`.

### What Does `SFTTrainer` + QLoRA / LoRA Modify?
1. **Inside Each Linear Layer (Frozen Base $W_0$ + 16-bit LoRA Bypass):**
```text
                        Output Activation h = h_base + (α/r) · h_lora
                                       ▲
                                       │
                                      (+)  ◄── Elementwise Sum in bfloat16
                                     ╱   ╲
         ┌──────────────────────────┘     └───────────────────────────┐
         │  h_base = W_0 · x                h_lora = (α/r) · B · (A · x)
┌────────┴────────────────────────┐          ┌────────────────────────┴────────┐
│   FROZEN BASE WEIGHT W_0        │          │   TRAINABLE LoRA ADAPTER (B·A)  │
│ • 4-bit NF4   (~1.4 GB) OR      │          │ • Matrix B [d_out × r=16] (bf16)│
│   16-bit bf16 (~5.0 GB)         │          │ • Matrix A [r=16 × d_in]  (bf16)│
│ • requires_grad = False         │          │ • requires_grad = True          │
│ • 0 optimizer states            │          │ • Updated by SFTTrainer AdamW   │
└────────▲────────────────────────┘          └────────────────▲────────────────┘
         └──────────────────────────┐     ┌───────────────────┘
                                     ╲   ╱
                             Input Activation x (bfloat16)
```

2. **Inside Each of the 26 Gemma-2-2B Transformer Blocks (7 Modified Linear Projections):**
```text
  Input Tokens ──► [ Token Embeddings (256,000 × 2304) ]  ──► FROZEN
                                │
        ┌───────────────────────▼───────────────────────┐
        │           LAYER l ∈ {1 ... 26}                │
        │  • Grouped-Query Attention (GQA):             │
        │     ├─► q_proj (2304 → 2048)  [+LoRA A, B] ◄──┼── MODIFIED (r=16)
        │     ├─► k_proj (2304 → 1024)  [+LoRA A, B] ◄──┼── MODIFIED (r=16)
        │     ├─► v_proj (2304 → 1024)  [+LoRA A, B] ◄──┼── MODIFIED (r=16)
        │     └─► o_proj (2048 → 2304)  [+LoRA A, B] ◄──┼── MODIFIED (r=16)
        │  • GeGLU MLP (Feed-Forward Network):          │
        │     ├─► gate_proj (2304 → 9216) [+LoRA A, B] ◄┼── MODIFIED (r=16)
        │     ├─► up_proj   (2304 → 9216) [+LoRA A, B] ◄┼── MODIFIED (r=16)
        │     └─► down_proj (9216 → 2304) [+LoRA A, B] ◄┼── MODIFIED (r=16)
        │  • All RMSNorms & Residuals ──────────────────┼──► FROZEN
        └───────────────────────┬───────────────────────┘
  TOTAL BASE PARAMETERS (FROZEN): 2,614,341,888 (99.21%)
  TOTAL LoRA PARAMETERS (TRAIN) :    20,766,720 ( 0.79%) = 798,720 × 26 layers
```

3. **Across the Token Sequence (`completion_only_loss=True` Masking):**
```text
Token Stream:   <bos> <start_of_turn>user\n ... [~860 Prompt Tokens] ... <start_of_turn>model\n {"labels": ["Zugfahrt:negative"]} <end_of_turn>\n
Target Labels:  -100         -100                        -100                            -100   "{"  "labels" ... "Zugfahrt:negative" "]}" <end_of_turn>
                  └───────────────────────────────────┬───────────────────────────────────┘      └────────────────────┬────────────────────┘
                                       IGNORED BY CROSS-ENTROPY LOSS                                   ACTIVE CROSS-ENTROPY LOSS
                                         (No Gradient Signal)                                        (100% of Backprop Updates A & B)
```

In [ ]:
import importlib
import germeval_llm.training
importlib.reload(germeval_llm.training)
from germeval_llm.training import run_sft_adapter_training

if RUN_LIVE_GPU:
    train_summary = run_sft_adapter_training(
        config_path=TRAIN_CONFIG,
        train_jsonl_path='data/derived/task_c/train.jsonl',
        dev_jsonl_path='data/derived/task_c/dev.jsonl',
        output_adapter_dir=ADAPTER_DIR,
        subset_size=TRAIN_SUBSET_SIZE,
        max_steps_override=MAX_TRAIN_STEPS,
        sampling_strategy='stratified',
        seed_override=42,
    )
    torch.cuda.empty_cache()
else:
    train_summary = json.loads(Path('results/runs/smoke_task_c_lora_train_summary.json').read_text())

print(f'Adapter Training Summary ({METHOD_LABEL}):')
print(json.dumps(train_summary, indent=2))

---
## Step 6: Evaluate the Fine-Tuned Adapter on `test_syn` & Compare Side-by-Side
Now we reload the saved adapter (`ADAPTER_DIR`) onto the base model using the same precision setting (`load_in_4bit=USE_4BIT_QUANTIZATION`), run greedy inference on the exact same `test_syn` split, and compare Before vs. After metrics in pure Python:

In [ ]:
if RUN_LIVE_GPU:
    ft_model, tokenizer = load_gemma_model_and_tokenizer(
        model_id='google/gemma-2-2b-it',
        adapter_path=ADAPTER_DIR,
        load_in_4bit=USE_4BIT_QUANTIZATION,
    )
    ft_gen_fn = build_hf_generator_fn(ft_model, tokenizer, max_new_tokens=256)
    ft_preds = run_task_c_inference_with_generator(
        eval_docs,
        run_id=f'colab_{METHOD_TAG}_r16',
        split='test_syn',
        generate_fn=ft_gen_fn,
        prompt_key='p1_guidelines',
    )
    save_predictions_jsonl(ft_preds, FT_PRED_PATH)
    ft_metrics = evaluate_task_c_predictions(eval_docs, ft_preds)
    del ft_model
    torch.cuda.empty_cache()
else:
    ft_preds = load_predictions_jsonl('results/predictions/smoke_lora_dev_dev.jsonl')
    ft_metrics = json.loads(Path('results/metrics/smoke_lora_dev_dev.json').read_text())

comparison_df = pd.DataFrame([
    {
        'Model': '1. Frozen Gemma-2-2B-IT (Zero-Shot P1)',
        'Valid Exact JSON Rate': baseline_metrics['valid_exact_rate'],
        'Schema Validity Rate': baseline_metrics['schema_validity_rate'],
        'Task C1 Strict Micro-F1': round(baseline_metrics['task_c1_category_strict']['micro_f1'], 4),
        'Task C2 Strict Micro-F1': round(baseline_metrics['task_c2_category_sentiment_strict']['micro_f1'], 4),
        'Task C2 Official Java-Compat F1': round(baseline_metrics['task_c2_category_sentiment_java_compat']['micro_f1'], 4),
        'Mean Latency (ms/doc)': round(baseline_metrics['efficiency']['mean_latency_ms_per_doc'], 1),
    },
    {
        'Model': f'2. Fine-Tuned Gemma-2-2B-IT ({METHOD_LABEL})',
        'Valid Exact JSON Rate': ft_metrics['valid_exact_rate'],
        'Schema Validity Rate': ft_metrics['schema_validity_rate'],
        'Task C1 Strict Micro-F1': round(ft_metrics['task_c1_category_strict']['micro_f1'], 4),
        'Task C2 Strict Micro-F1': round(ft_metrics['task_c2_category_sentiment_strict']['micro_f1'], 4),
        'Task C2 Official Java-Compat F1': round(ft_metrics['task_c2_category_sentiment_java_compat']['micro_f1'], 4),
        'Mean Latency (ms/doc)': round(ft_metrics['efficiency']['mean_latency_ms_per_doc'], 1),
    },
])
display(comparison_df)

---
## Step 7: Qualitative Inspection of Predictions Before vs. After Fine-Tuning
Let's inspect the raw strings emitted by the base model vs. the fine-tuned adapter on individual documents:

In [ ]:
from germeval_llm.convert import extract_task_c_label_sets

docs_slice = eval_docs[:5] if RUN_LIVE_GPU else dev_docs[:5]
b_slice = baseline_preds[:5]
q_slice = ft_preds[:5]

for doc, bp, qp in zip(docs_slice, b_slice, q_slice):
    gold_c2, _ = extract_task_c_label_sets(doc, apply_typo_patches=True)
    print('=' * 90)
    print('Text:', doc.text[:160])
    print('Gold C2 Labels :', gold_c2)
    print(f'Zero-Shot Raw  ({bp.parse_status}): {bp.raw_output!r}')
    print(f'Fine-Tuned Raw ({qp.parse_status}): {qp.raw_output!r}')